# The Poisson distribution
The PMF by hand and with scipy, interval probabilities, mean = variance by simulation, and the binomial-to-Poisson limit.

In [1]:
import math
import numpy as np
import plotly.graph_objects as go
from scipy import stats

## 1. Exactly 7 questions when 4 are usual (lambda = 4)

In [2]:
lam, y = 4, 7
# lambda^y * e^(-lambda) / y!
by_hand = lam ** y * math.exp(-lam) / math.factorial(y)
print(lam ** y, round(math.exp(-lam), 4), math.factorial(y))
print('by hand:', round(by_hand, 4), ' scipy:', round(stats.poisson.pmf(y, lam), 4))

16384 0.0183 5040
by hand: 0.0595  scipy: 0.0595


## 2. The whole PMF: it adds to 1

In [3]:
questions = stats.poisson(4)
for k in range(13):
    print(k, round(questions.pmf(k), 4))
# the counts never stop, but the far tail is tiny
print('sum of P(0..50):', questions.pmf(np.arange(51)).sum())

0 0.0183
1 0.0733
2 0.1465
3 0.1954
4 0.1954
5 0.1563
6 0.1042
7 0.0595
8 0.0298
9 0.0132
10 0.0053
11 0.0019
12 0.0006
sum of P(0..50): 0.9999999999999999


## 3. Probabilities of intervals: add the bars

In [4]:
print('P(Y >= 7)     =', round(questions.sf(6), 4))     # 1 - P(Y <= 6)
print('P(Y <= 2)     =', round(questions.cdf(2), 4))
print('P(2 <= Y <= 6) =', round(questions.cdf(6) - questions.cdf(1), 4))
print('same by adding bars:', round(questions.pmf(range(2, 7)).sum(), 4))

P(Y >= 7)     = 0.1107
P(Y <= 2)     = 0.2381
P(2 <= Y <= 6) = 0.7977
same by adding bars: 0.7977


## 4. Fireflies: 3 flashes per 10 seconds, so 6 per 20 seconds

In [5]:
rate_per_second = 3 / 10
lam_20 = rate_per_second * 20       # scale the rate to the new interval
print('lambda for 20 s:', lam_20)
print('P(8 flashes in 20 s) =', round(stats.poisson.pmf(8, lam_20), 4))

lambda for 20 s: 6.0
P(8 flashes in 20 s) = 0.1033


## 5. Mean and variance are both lambda

In [6]:
rng = np.random.default_rng(42)     # fixed seed
days = rng.poisson(lam=4, size=100_000)
print('first days:', days[:10])
print('average', days.mean(), ' variance', round(days.var(), 4))
print('scipy: mean', questions.mean(), ' variance', questions.var(), ' std', questions.std())

first days: [6 3 6 8 1 6 2 7 3 0]
average 3.99437  variance 3.9874
scipy: mean 4.0  variance 4.0  std 2.0


In [7]:
# Overdispersion: break one Poisson condition at a time, keep the mean near 4,
# and compare P(Y >= 12) with what a Poisson model with the same mean predicts.
rng = np.random.default_rng(42)
cases = {
    'constant rate, independent': rng.poisson(4, size=100_000),
    'rate varies from day to day': rng.poisson(rng.gamma(shape=2, scale=2, size=100_000)),
    'events come in pairs': 2 * rng.poisson(2, size=100_000),
}
for name, y in cases.items():
    print(f'{name:28} mean {y.mean():.2f}  variance {y.var():.2f}  '
          f'P(Y>=12) seen {(y >= 12).mean():.4f}  Poisson says {stats.poisson(y.mean()).sf(11):.4f}')


constant rate, independent   mean 3.99  variance 3.99  P(Y>=12) seen 0.0008  Poisson says 0.0009
rate varies from day to day  mean 3.99  variance 11.99  P(Y>=12) seen 0.0382  Poisson says 0.0009
events come in pairs         mean 4.01  variance 7.96  P(Y>=12) seen 0.0163  Poisson says 0.0009


## 6. Binomial with large n and small p is close to Poisson

In [8]:
k = np.arange(30)
for n, p in [(10, 0.4), (40, 0.1), (1000, 0.004)]:
    gap = np.abs(stats.binom.pmf(k, n, p) - stats.poisson.pmf(k, n * p)).max()
    print(f'n = {n:5}, p = {p:5}: largest gap {gap:.4f}')
# the rule of thumb n >= 20, p <= 0.05, at its edge and beyond it
for n, p in [(20, 0.05), (100, 0.05), (20, 0.2)]:
    gap = np.abs(stats.binom.pmf(k, n, p) - stats.poisson.pmf(k, n * p)).max()
    print(f'n = {n:5}, p = {p:5}: largest gap {gap:.4f}')


n =    10, p =   0.4: largest gap 0.0555
n =    40, p =   0.1: largest gap 0.0105
n =  1000, p = 0.004: largest gap 0.0004
n =    20, p =  0.05: largest gap 0.0095
n =   100, p =  0.05: largest gap 0.0046
n =    20, p =   0.2: largest gap 0.0228


## 7. Try it: drag the slider to change lambda

In [9]:
y = np.arange(0, 26)
fig = go.Figure()
lams = [0.5, 1, 2, 4, 7, 10, 15]
for lam in lams:
    fig.add_bar(x=y, y=stats.poisson.pmf(y, lam), visible=(lam == 4), name=f'λ = {lam}')
steps = [dict(method='update', label=str(lam), args=[{'visible': [l == lam for l in lams]}]) for lam in lams]
fig.update_layout(sliders=[dict(active=3, currentvalue=dict(prefix='λ = '), steps=steps)],
                  template='simple_white', xaxis_title='count, y', yaxis_title='P(Y = y)')
fig.show()

## Cutting the day finer

P(7 questions) with the day cut into hours and into minutes, against the Poisson value; and the factors of the derivation at n = 1000, k = 7.

In [ ]:
import math
from scipy import stats
print('hours  ', stats.binom.pmf(7, 24, 4 / 24).round(4))
print('minutes', stats.binom.pmf(7, 1440, 4 / 1440).round(4))
print('Poisson', stats.poisson.pmf(7, 4).round(4))
n, lam, k = 1000, 4, 7
print('first factor ', round(math.prod(n - i for i in range(k)) / n ** k, 3))
print('third factor ', round((1 - lam / n) ** n, 4), ' e^-4 =', round(math.exp(-4), 4))
print('fourth factor', round((1 - lam / n) ** (-k), 3))